In [ ]:
year = '2022'
nmonths = 6
minrate = -2
maxrate = 8
ngrid = 7000
lam_param = 0.01
source = 'LEI'

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy.stats import norm
import scipy.optimize as opt
from scipy import stats
import matplotlib.pyplot as plt
from scipy.optimize import minimize 
from scipy.interpolate import make_smoothing_spline, make_interp_spline, splev, griddata
import warnings
import os
import traceback  
warnings.filterwarnings("ignore")

In [ ]:
def compute_expiry_date_closest(date,nmonths):
    add_date = date + relativedelta(months = nmonths)
    output_month = add_date.month
    selected_months = np.array([3, 6, 9, 12, 15])
    diff = abs(selected_months - output_month)
    add_months = output_month - selected_months[np.argmin(diff)]
    output = add_date - relativedelta(months = add_months)
    return(output)

def implvol_bs(sigma,S,K,P,tau,r,q):
    done = (np.log(S/K) + tau*(r-q+(sigma**2)/2))/(sigma*np.sqrt(tau))
    dtwo = done - sigma*np.sqrt(tau)
    return (S*np.exp(-q*tau))*norm.cdf(done) - (K*np.exp(-r*tau))*norm.cdf(dtwo) - P

def price_bs(sigma,S,K,tau,r,q):
    done = (np.log(S/K) + tau*(r-q+(sigma**2)/2))/(sigma*np.sqrt(tau))
    dtwo = done - sigma*np.sqrt(tau)
    return (S*np.exp(-q*tau))*norm.cdf(done) - (K*np.exp(-r*tau))*norm.cdf(dtwo)

def delta_bs(S,r,q,tau,K,sigma):
    done = (np.log(S/K) + tau*(r-q+(sigma**2)/2))/(sigma*np.sqrt(tau))
    return norm.cdf(done)*np.exp(-q*tau)

def vega_bs(S,K,sigma,tau,r,q):
    done = (np.log(S/K) + tau*(r-q+(sigma**2)/2))/(sigma*np.sqrt(tau))
    return S*norm.pdf(done)*np.sqrt(tau)*np.exp(-q*tau)
    
def k_from_delta_bs(S,r,q,tau,sigma,delta):
    return S/np.exp(norm.ppf(delta*np.exp(q*tau))*sigma*np.sqrt(tau) - (r-q+sigma**2/2)*tau)

def put_call_parity(putP,S,K,q,r,tau):
    return putP + S*np.exp(-q*tau) - K*np.exp(-r*tau)

def poly(x,y,p):
    return sum((y-(p[0]*x**3 + p[1]*x**2 + p[2]*x + p[3]))**2) 
    
def cubic(x,p):
    return p[0]*x**3 + p[1]*x**2 + p[2]*x + p[3]

In [ ]:
if source == 'GQE':
    data_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\RiskNeutralDensities\output\options_' + year + '.csv'
elif source == 'LEI':
    data_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\RiskNeutralDensities\output\options_' + year + '_LEI' +'.csv'
elif source == 'MP':
    data_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\RiskNeutralDensities\output\options_' + year + '_MP.csv'
options_data = pd.read_csv(data_path)
for i in options_data.index:
    options_data.loc[i,'time'] = datetime.strptime(str(options_data.loc[i,'time']),'%Y%m%d')
    options_data.loc[i,'exp_date'] = datetime.strptime(str(options_data.loc[i,'exp_date']),'%Y%m%d')
    options_data.loc[i,'diff'] = options_data.loc[i,'diff']/365

time = sorted(list(set(options_data.time)))

In [ ]:
for day in time:
    date = day
    date_str = date.strftime('%d-%m-%Y')
    expiry = compute_expiry_date_closest(date=date,nmonths=nmonths)
    date_str = date.strftime('%d-%m-%Y')
    ninterp = 1000
    print(date.strftime('%d-%m-%Y'))
    print('Main Expiry', expiry)
    
    lb = datetime(expiry.year,expiry.month,1)
    ub = lb + relativedelta(months=1) - relativedelta(days = 1)
    try:
        options = options_data.loc[(options_data['time'] == date)  & (options_data['exp_date'] >= lb) & (options_data['exp_date'] <= ub)]
        options.reset_index(drop=True,inplace=True)
        options.columns = ['Time','Expiry','tau','Class','S','K','P']
        
        # Do not interpolate for options close (15 days) to 3 month maturity
        tau_ub = (365/12*nmonths+15)/365
        tau_lb = (365/12*nmonths-15)/365
        if options.tau[0] > tau_ub:
            expiry_interp = expiry - relativedelta(months=3)
        elif options.tau[0] < tau_lb:
            expiry_interp = expiry + relativedelta(months=3)
        else:
            expiry_interp = expiry
        
        riskfree = 0
        dividend = 0
        print('Secondary Expiry', expiry_interp)
        ##########################################################
        ### Compute Delta-Sigma Interpolation for Main Expiry Date
        ##########################################################
        lb = datetime(expiry.year,expiry.month,1)
        ub = lb + relativedelta(months=1) - relativedelta(days = 1)
        options = options_data.loc[(options_data['time'] == date)  & (options_data['exp_date'] >= lb) & (options_data['exp_date'] <= ub)]
        options.reset_index(drop=True,inplace=True)
        options.columns = ['Time','Expiry','tau','Class','S','K','P']
        tau_main = options.tau[0]
        s_main = options.S[0]
        
        # Keep traded options only
        options = options.loc[options['P'] > 0]
        
        ### Call Options
        options_call = options.loc[options['Class'] == 1]
        options_call.reset_index(inplace=True,drop=True)
        options_call.sort_values(by='K',ascending=True,inplace=True)
        # Check Convexity
        while len(options_call) > 3:
            convexity = list()
            convexity.append(True)
            for i in range(1,len(options_call)-1,1):
                triple = options_call.iloc[i-1:i+2]
                triple.reset_index(inplace=True,drop=True)
                beta = (triple.iloc[2].P - triple.iloc[0].P)/(triple.iloc[2].K - triple.iloc[0].K)
                alpha = triple.iloc[2].P - beta*triple.iloc[2].K
                convexity.append(triple.iloc[1].P <=( beta*triple.iloc[1].K + alpha))
            convexity.append(True)
            options_call['Convexity'] = convexity
            if sum(options_call.Convexity) == len(options_call):
                break
            else:
                options_call = options_call.loc[options_call['Convexity'] == True]
                options_call.reset_index(inplace=True,drop=True)
        # Check Monotonicity
        while len(options_call) > 2:
            monotonicity = list()
            monotonicity.append(True)
            for i in range(1,len(options_call),1):
                double = options_call.iloc[i-1:i+1]
                double.reset_index(inplace=True,drop=True)
                monotonicity.append((double.iloc[1].P - double.iloc[0].P)/(double.iloc[1].K - double.iloc[0].K) <= 0)
            options_call['Monotonicity'] = monotonicity
            if sum(options_call.Monotonicity) == len(options_call):
                break
            else:
                options_call = options_call.loc[options_call['Monotonicity'] == True]
                options_call.reset_index(inplace=True,drop=True)
        
        ### Put Options
        options_put = options.loc[options['Class'] == 2]
        options_put.reset_index(inplace=True,drop=True)
        options_put.sort_values(by='K',ascending=True,inplace=True)
        # Check Convexity
        while len(options_put) > 3:
            convexity = list()
            convexity.append(True)
            for i in range(1,len(options_put)-1,1):
                triple = options_put.iloc[i-1:i+2]
                triple.reset_index(inplace=True,drop=True)
                beta = (triple.iloc[2].P - triple.iloc[0].P)/(triple.iloc[2].K - triple.iloc[0].K)
                alpha = triple.iloc[2].P - beta*triple.iloc[2].K
                convexity.append(triple.iloc[1].P <=( beta*triple.iloc[1].K + alpha))
            convexity.append(True)
            options_put['Convexity'] = convexity
            if sum(options_put.Convexity) == len(options_put):
                break
            else:
                options_put = options_put.loc[options_put['Convexity'] == True]
                options_put.reset_index(inplace=True,drop=True)
        # Check Monotonicity
        while len(options_put) > 2:
            monotonicity = list()
            monotonicity.append(True)
            for i in range(1,len(options_put),1):
                double = options_put.iloc[i-1:i+1]
                double.reset_index(inplace=True,drop=True)
                monotonicity.append((double.iloc[0].P - double.iloc[1].P)/(double.iloc[0].K - double.iloc[1].K) >= 0)
            options_put['Monotonicity'] = monotonicity
            if sum(options_put.Monotonicity) == len(options_put):
                break
            else:
                options_put = options_put.loc[options_put['Monotonicity'] == True]
                options_put.reset_index(inplace=True,drop=True)
        # Call Price by Put-Call Parity
        put_call = list()
        for i in options_put.index:
            P = options_put.loc[i].P
            S = options_put.loc[i].S
            K = options_put.loc[i].K
            tau = options_put.loc[i].tau
            put_call.append(put_call_parity(P,S,K,dividend,riskfree,tau))
        options_put['P'] = put_call
        
        ### Merge Call and Put Options
        options_final_main = pd.concat([options_call,options_put],axis=0)
        options_final_main = options_final_main.loc[options_final_main['P'] > 0]
        options_final_main.reset_index(inplace=True,drop=True)
        
        # Implied Volatility
        impvol = list()
        for i in options_final_main.index:
            S = options_final_main.loc[i].S
            K = options_final_main.loc[i].K
            P = options_final_main.loc[i].P
            tau = options_final_main.loc[i].tau
            try:
                impvol.append(opt.root_scalar(lambda sigma: implvol_bs(sigma,S,K,P,tau,riskfree,dividend),x0=0.03,method='newton').root)
            except:
                impvol.append(np.nan)
        options_final_main['Sigma'] = impvol
        # Keep options_put with positive implied volatility only
        options_final_main = options_final_main.loc[options_final_main['Sigma'] > 0]
        
        #  Implied Delta
        delta = list()
        for i in options_final_main.index:
            S = options_final_main.loc[i].S
            K = options_final_main.loc[i].K
            P = options_final_main.loc[i].P
            tau = options_final_main.loc[i].tau
            sigma = options_final_main.loc[i].Sigma
            try:
                delta.append(round(delta_bs(S,riskfree,dividend,tau,K,sigma),6))
            except:
                delta.append(np.nan)
        options_final_main['Delta'] = delta
        # Keep options_put with Delta between 0.01 and 0.99
        options_final_main = options_final_main.loc[(options_final_main['Delta'] >= 0.01) & (options_final_main['Delta'] <= 0.99)]
        
        # Implied Vega
        vega = list()
        for i in options_final_main.index:
            S = options_final_main.loc[i].S
            K = options_final_main.loc[i].K
            sigma = options_final_main.loc[i].Sigma
            tau = options_final_main.loc[i].tau
            r = riskfree
            q = dividend
            vega.append(vega_bs(S,K,sigma,tau,r,q))
        options_final_main['Vega'] = vega
        
        ### Remove Implied Volatility Outliers
        options_final_main.sort_values(by='Delta',inplace=True)
        options_final_main.drop_duplicates(subset=['Delta'],inplace=True)
        options_final_main.reset_index(drop=True,inplace=True)
        try:
            sigma = options_final_main.Sigma
            z = abs(stats.zscore(sigma))
            z = z.loc[z < norm.ppf(0.99)]
            options_final_main = options_final_main.loc[z.index]
            options_final_main.reset_index(drop=True,inplace=True)
            # Remove Points much far from Smile
            len_before = len(options_final_main)
            len_after = len_before+1
            while len_before != len_after:
                len_before = len(options_final_main)
                m = options_final_main.Sigma.diff()/options_final_main.Delta.diff()
                index_rem = list()
                for i  in range(1,len(m)-1):
                    if (np.sign(m[i]) != np.sign(m[i-1])) & (np.sign(m[i]) != np.sign(m[i+1])):
                        index_rem.append(i-1)
                options_final_main.drop(index_rem,inplace=True)
                options_final_main.reset_index(drop=True,inplace=True)
                len_after = len(options_final_main)
        except:
            options_final_main = options_final_main
      
        ###############################################################
        ### Compute Delta-Sigma Interpolation for Secondary Expiry Date
        ###############################################################
        lb = datetime(expiry_interp.year,expiry_interp.month,1)
        ub = lb + relativedelta(months=1) - relativedelta(days = 1)
        
        options = options_data.loc[(options_data['time'] == date)  & (options_data['exp_date'] >= lb) & (options_data['exp_date'] <= ub)]
        options.reset_index(drop=True,inplace=True)
        options.columns = ['Time','Expiry','tau','Class','S','K','P']
        tau_secondary = options.tau[0]
        s_secondary = options.S[0]
        
        # Keep traded options only
        options = options.loc[options['P'] > 0]
        
        ### Call Options
        options_call = options.loc[options['Class'] == 1]
        options_call.reset_index(inplace=True,drop=True)
        options_call.sort_values(by='K',ascending=True,inplace=True)
        # Check Convexity
        while len(options_call) > 3:
            convexity = list()
            convexity.append(True)
            for i in range(1,len(options_call)-1,1):
                triple = options_call.iloc[i-1:i+2]
                triple.reset_index(inplace=True,drop=True)
                beta = (triple.iloc[2].P - triple.iloc[0].P)/(triple.iloc[2].K - triple.iloc[0].K)
                alpha = triple.iloc[2].P - beta*triple.iloc[2].K
                convexity.append(triple.iloc[1].P <=( beta*triple.iloc[1].K + alpha))
            convexity.append(True)
            options_call['Convexity'] = convexity
            if sum(options_call.Convexity) == len(options_call):
                break
            else:
                options_call = options_call.loc[options_call['Convexity'] == True]
                options_call.reset_index(inplace=True,drop=True)
        # Check Monotonicity
        while len(options_call) > 2:
            monotonicity = list()
            monotonicity.append(True)
            for i in range(1,len(options_call),1):
                double = options_call.iloc[i-1:i+1]
                double.reset_index(inplace=True,drop=True)
                monotonicity.append((double.iloc[1].P - double.iloc[0].P)/(double.iloc[1].K - double.iloc[0].K) <= 0)
            options_call['Monotonicity'] = monotonicity
            if sum(options_call.Monotonicity) == len(options_call):
                break
            else:
                options_call = options_call.loc[options_call['Monotonicity'] == True]
                options_call.reset_index(inplace=True,drop=True)
        
        ### Put Options
        options_put = options.loc[options['Class'] == 2]
        options_put.reset_index(inplace=True,drop=True)
        options_put.sort_values(by='K',ascending=True,inplace=True)
        # Check Convexity
        while len(options_put) > 3:
            convexity = list()
            convexity.append(True)
            for i in range(1,len(options_put)-1,1):
                triple = options_put.iloc[i-1:i+2]
                triple.reset_index(inplace=True,drop=True)
                beta = (triple.iloc[2].P - triple.iloc[0].P)/(triple.iloc[2].K - triple.iloc[0].K)
                alpha = triple.iloc[2].P - beta*triple.iloc[2].K
                convexity.append(triple.iloc[1].P <=( beta*triple.iloc[1].K + alpha))
            convexity.append(True)
            options_put['Convexity'] = convexity
            if sum(options_put.Convexity) == len(options_put):
                break
            else:
                options_put = options_put.loc[options_put['Convexity'] == True]
                options_put.reset_index(inplace=True,drop=True)
        # Check Monotonicity
        while len(options_put) > 2:
            monotonicity = list()
            monotonicity.append(True)
            for i in range(1,len(options_put),1):
                double = options_put.iloc[i-1:i+1]
                double.reset_index(inplace=True,drop=True)
                monotonicity.append((double.iloc[0].P - double.iloc[1].P)/(double.iloc[0].K - double.iloc[1].K) >= 0)
            options_put['Monotonicity'] = monotonicity
            if sum(options_put.Monotonicity) == len(options_put):
                break
            else:
                options_put = options_put.loc[options_put['Monotonicity'] == True]
                options_put.reset_index(inplace=True,drop=True)
                
        # Call Price by Put-Call Parity
        put_call = list()
        for i in options_put.index:
            P = options_put.loc[i].P
            S = options_put.loc[i].S
            K = options_put.loc[i].K
            tau = options_put.loc[i].tau
            put_call.append(put_call_parity(P,S,K,dividend,riskfree,tau))
        options_put['P'] = put_call
        
        ### Merge Call and Put Options
        options_final_secondary = pd.concat([options_call,options_put],axis=0)
        options_final_secondary = options_final_secondary.loc[options_final_secondary['P'] > 0]
        options_final_secondary.reset_index(inplace=True,drop=True)
        
        # Implied Volatility
        impvol = list()
        for i in options_final_secondary.index:
            S = options_final_secondary.loc[i].S
            K = options_final_secondary.loc[i].K
            P = options_final_secondary.loc[i].P
            tau = options_final_secondary.loc[i].tau
            try:
                impvol.append(opt.root_scalar(lambda sigma: implvol_bs(sigma,S,K,P,tau,riskfree,dividend),x0=0.03,method='newton').root)
            except:
                impvol.append(np.nan)
        options_final_secondary['Sigma'] = impvol
        # Keep options_put with positive implied volatility only
        options_final_secondary = options_final_secondary.loc[options_final_secondary['Sigma'] > 0]
        
        #  Implied Delta
        delta = list()
        for i in options_final_secondary.index:
            S = options_final_secondary.loc[i].S
            K = options_final_secondary.loc[i].K
            P = options_final_secondary.loc[i].P
            tau = options_final_secondary.loc[i].tau
            sigma = options_final_secondary.loc[i].Sigma
            try:
                delta.append(round(delta_bs(S,riskfree,dividend,tau,K,sigma),6))
            except:
                delta.append(np.nan)
        options_final_secondary['Delta'] = delta
        # Keep options_put with Delta between 0.01 and 0.99
        options_final_secondary = options_final_secondary.loc[(options_final_secondary['Delta'] >= 0.01) & (options_final_secondary['Delta'] <= 0.99)]
        
        # Implied Vega
        vega = list()
        for i in options_final_secondary.index:
            S = options_final_secondary.loc[i].S
            K = options_final_secondary.loc[i].K
            sigma = options_final_secondary.loc[i].Sigma
            tau = options_final_secondary.loc[i].tau
            r = riskfree
            q = dividend
            vega.append(vega_bs(S,K,sigma,tau,r,q))
        options_final_secondary['Vega'] = vega
        
        ### Remove Implied Volatility Outliers
        options_final_secondary.sort_values(by='Delta',inplace=True)
        options_final_secondary.drop_duplicates(subset=['Delta'],inplace=True)
        options_final_secondary.reset_index(drop=True,inplace=True)
        try:
            sigma = options_final_secondary.Sigma
            z = abs(stats.zscore(sigma))
            z = z.loc[z < norm.ppf(0.99)]
            options_final_secondary = options_final_secondary.loc[z.index]
            options_final_secondary.reset_index(drop=True,inplace=True)
            # Remove Points much far from Smile
            len_before = len(options_final_secondary)
            len_after = len_before+1
            while len_before != len_after:
                len_before = len(options_final_secondary)
                m = options_final_secondary.Sigma.diff()/options_final_secondary.Delta.diff()
                index_rem = list()
                for i  in range(1,len(m)-1):
                    if (np.sign(m[i]) != np.sign(m[i-1])) & (np.sign(m[i]) != np.sign(m[i+1])):
                        index_rem.append(i-1)
                options_final_secondary.drop(index_rem,inplace=True)
                options_final_secondary.reset_index(drop=True,inplace=True)
                len_after = len(options_final_secondary)
        except:
            options_final_secondary = options_final_secondary

        ################################
        ### Interpolate Volatility Smile
        ################################
        print('Options in Main Set:', len(options_final_main))
        print('Options in Secondary Set:', len(options_final_secondary))
        # When len(set) < 5 interpolation cannot be performed. Try and replace inadequate set.
        if (len(options_final_main) >= 5 ) & (len(options_final_secondary) < 5):
            options_final_secondary = options_final_main
        if (len(options_final_secondary) >= 5 ) & (len(options_final_main) < 5):
            options_final_main = options_final_secondary     
        
        # Main Set
        x = options_final_main.Delta
        y = options_final_main.Sigma
        w = options_final_main.Vega
       # gridDelta_red = np.linspace(min(x),max(x),ninterp)
        gridDelta_red = np.linspace(0.01,0.99,ninterp)
        spline = make_smoothing_spline(x, y, w=w, lam=lam_param)
        interp_sigma_red = splev(gridDelta_red, spline)
        options_interpolate_main_spline = pd.DataFrame(data={'delta': gridDelta_red, 'sigma': interp_sigma_red,
                                                             'tau': np.ones(len(gridDelta_red))*tau_main,'s': np.ones(len(gridDelta_red))*s_main})
        # gridDelta = np.linspace(0.01,0.99,ninterp)
        # cons = ({'type': 'ineq', 'fun': lambda p: cubic(gridDelta,p)})
        # coeff = minimize(lambda p: poly(gridDelta_red,interp_sigma_red,p), (0,0,0,0), method='SLSQP',constraints=cons).x
        # interp_sigma = cubic(gridDelta,coeff)
        # options_interpolate_main = pd.DataFrame(data={'delta': gridDelta, 'sigma': interp_sigma,
        #                                                    'tau': np.ones(len(gridDelta))*tau_main,'s': np.ones(len(gridDelta))*s_main})
        options_interpolate_main = options_interpolate_main_spline
        
        # Secondary Set
        x = options_final_secondary.Delta
        y = options_final_secondary.Sigma
        w = options_final_secondary.Vega
        # gridDelta_red = np.linspace(min(x),max(x),ninterp)
        gridDelta_red = np.linspace(0.01,0.99,ninterp)
        spline = make_smoothing_spline(x, y, w=w, lam=lam_param)
        interp_sigma_red = splev(gridDelta_red, spline)  
        options_interpolate_secondary_spline = pd.DataFrame(data={'delta': gridDelta_red, 'sigma': interp_sigma_red,
                                                                  'tau': np.ones(len(gridDelta_red))*tau_secondary,'s': np.ones(len(gridDelta_red))*s_secondary})
        # gridDelta = np.linspace(0.01,0.99,ninterp)
        # cons = ({'type': 'ineq', 'fun': lambda p: cubic(gridDelta,p)})
        # coeff = minimize(lambda p: poly(gridDelta_red,interp_sigma_red,p), (0,0,0,0), method='SLSQP',constraints=cons).x
        # interp_sigma = cubic(gridDelta,coeff)
        # options_interpolate_secondary = pd.DataFrame(data={'delta': gridDelta, 'sigma': interp_sigma,
        #                                                    'tau': np.ones(len(gridDelta))*tau_secondary,'s': np.ones(len(gridDelta))*s_secondary})
        options_interpolate_secondary = options_interpolate_secondary_spline
        gridDelta = gridDelta_red
        
    #############################################################################
    ### Evaluate Implied PDF at constant maturituy of 3 months
    #############################################################################
        if expiry != expiry_interp:
            ### Interpolate Delta-Sigma space across tau to get constant maturity tau=nmonths/12
            m = (options_interpolate_main.sigma - options_interpolate_secondary.sigma)/(options_interpolate_main.tau - options_interpolate_secondary.tau)
            c = options_interpolate_main.sigma - m*options_interpolate_main.tau
            sigma_constant_maturity = c + m*(nmonths/12)
            m = (options_interpolate_main.s - options_interpolate_secondary.s)/(options_interpolate_main.tau - options_interpolate_secondary.tau)
            c = options_interpolate_main.s - m*options_interpolate_main.tau
            s_constant_maturity = c + m*(nmonths/12)
            options_interpolate = pd.DataFrame(data={'delta':gridDelta, 'sigma': sigma_constant_maturity,
                                                     'tau': np.ones(len(gridDelta))*(nmonths/12),'s':s_constant_maturity})
        else:
            options_interpolate = pd.DataFrame(data={'delta':gridDelta, 'sigma': options_interpolate_main.sigma,
                                                     'tau': np.ones(len(gridDelta))*tau_main,'s':np.ones(len(gridDelta))*s_main})

        ### Retrieve K from delta
        K = list()
        for i in options_interpolate.index:
            sigma = options_interpolate.loc[i].sigma
            delta = options_interpolate.loc[i].delta
            S = options_interpolate.loc[i].s
            tau = options_interpolate.loc[i].tau
            K.append(k_from_delta_bs(S,riskfree,dividend,tau,sigma,delta))
        options_interpolate['K'] = K  
        
        options_interpolate = options_interpolate.loc[(options_interpolate.K > 93) & (options_interpolate.K < 102)]
        
        ### Compute Option Prices from Interpolated Volatility Smile
        price = list()
        for i in options_interpolate.index:
            sigma = options_interpolate.loc[i,'sigma']
            K = options_interpolate.loc[i,'K']
            S = options_interpolate.loc[i,'s']
            tau = options_interpolate.loc[i,'tau']    
            try:
                price.append(max(0,price_bs(sigma,S,K,tau,riskfree,dividend)))
            except:
                price.append(np.nan) 
        d = {'K': options_interpolate['K'], 'C': price}
        options_interp = pd.DataFrame(data=d)
        options_interp.sort_values(by='K',ascending=True,inplace=True)
        options_interp.reset_index(inplace=True,drop=True)
        for i in range(len(options_interp)-1):
            if options_interp.loc[i+1,'C'] > options_interp.loc[i,'C']:
                options_interp.loc[i+1,'C'] = options_interp.loc[i,'C']
        
        ### Compute Implied PDF
        pdf = list()
        for i in range(1,len(options_interp)-1,1):
            triple = options_interp[i-1:i+2]
            triple.reset_index(inplace=True,drop=True)
            z = np.polyfit(triple.K, triple.C, 3)
            p = np.poly1d(z)
            d2 = np.polyder(p,m=2)
            curvature = d2(triple.iloc[1].K) * np.exp(riskfree*tau)
            pdf.append(max(0,curvature))
        
        implied_pdf = pd.DataFrame(data={'Rate':100-options_interp.iloc[1:-1].K, date_str: pdf})
        implied_pdf.sort_values(by='Rate',inplace=True)
        implied_pdf.reset_index(inplace=True,drop=True)
        # Extrapolate Right Tail
        m = implied_pdf.diff().iloc[-1][date_str]/implied_pdf.diff().iloc[-1]['Rate']
        c = implied_pdf.iloc[-2][date_str] - m*implied_pdf.iloc[-2]['Rate']
        line = np.linspace(implied_pdf.iloc[-2]['Rate'],maxrate,100)
        line_right = c + m*line
        right = pd.DataFrame(data={'Rate':line, date_str: line_right})
        # Extrapolate Left Tail
        m = (implied_pdf.iloc[0:2][date_str].diff()/implied_pdf.iloc[0:2]['Rate'].diff()).iloc[-1]
        c = implied_pdf.iloc[1][date_str] - m*implied_pdf.iloc[1]['Rate']
        line = np.linspace(minrate,implied_pdf.iloc[2]['Rate'],100)
        line_left = c + m*line
        left = pd.DataFrame(data={'Rate':line, date_str: line_left})
        # Create Meaningful PDF
        implied_pdf = pd.concat([left,implied_pdf,right],axis=0)
        implied_pdf.loc[implied_pdf[date_str] < 0, date_str] = 0
        implied_pdf[date_str] = implied_pdf[date_str]/implied_pdf[date_str].sum()
        implied_pdf.sort_values(by='Rate',inplace=True)
        implied_pdf.drop_duplicates(subset='Rate',inplace=True)
        implied_pdf.reset_index(inplace=True,drop=True)
        # #  Create a Uniform Grid of Rates
        rategrid = np.linspace(minrate,maxrate,ngrid)
        pdf = griddata(implied_pdf.Rate, implied_pdf[date_str], rategrid, method='linear')
        implied_pdf = pd.DataFrame(data={'Rate':rategrid, date_str: pdf})
        
        ### Plots
        fig = plt.figure(figsize=(4,3))
        ax1 = fig.add_subplot(1, 1, 1)
        ax1.plot(options_interpolate_main.delta, options_interpolate_main.sigma,color='blue',label='Main')
        ax1.scatter(options_final_main.Delta, options_final_main.Sigma,color='blue')
        ax1.plot(options_interpolate_secondary.delta, options_interpolate_secondary.sigma,color='red',label='Secondary')
        ax1.scatter(options_final_secondary.Delta, options_final_secondary.Sigma,color='red')
        ax1.set_ylabel(r'$\sigma$')
        ax1.set_xlabel(r'$\delta$')
        ax1.legend()
        plt.show()

    except Exception as error:
        print(traceback.format_exc())
        rategrid = np.linspace(minrate,maxrate,ngrid)
        implied_pdf = pd.DataFrame(data={'Rate':rategrid, date_str: np.zeros(ngrid)})
            
    fig,ax1 = plt.subplots(1,1,figsize=(4,3))
    ax1.set_title('Option Implied PDF' + ' ' + date_str)
    ax1.plot(implied_pdf.Rate,implied_pdf[date_str])
    ax1.set_xlabel('i')
    ax1.set_ylabel(r'f(i)')
    ax1.axhline(0,color='black')
    plt.show()
    
    if date == time[0]:
        options_implied_pdfs = implied_pdf
    else:
        options_implied_pdfs = pd.concat([options_implied_pdfs, implied_pdf[date_str]],axis=1)

In [ ]:
if source == 'LEI':
    options_implied_pdfs.to_excel('Implied_Densities_' + year + '_LEI.xlsx',sheet_name=year,index=False,header=True)
elif source == 'MP':
    options_implied_pdfs.to_excel('Implied_Densities_' + year + '_MP.xlsx',sheet_name=year,index=False,header=True)
elif source == 'GQE':
    options_implied_pdfs.to_excel('Implied_Densities_' + year + '_GQE.xlsx',sheet_name=year,index=False,header=True)